# TP 1 — Premières transformations DataFrame
**Durée indicative : 40–55 min**

Objectifs : créer un DataFrame, renommer et ajouter une colonne, filtrer et agréger, écrire des partitions Parquet.

> **Environment:** Databricks notebook in a Git folder. Attach a compute resource with PySpark available. Data files are in the adjacent `data/` folder. This lab uses only **synthetic** data.

> **Expected skill level:** first experience with Big Data / Spark. Follow the cells in order.


In [ ]:
from pyspark.sql import functions as F
data = [("James", "Smith", "USA", 1, 200), ("Michael", "Rose", "FR", 2, 4000), ("Robert", "Williams", "FR", 3, 50), ("Maria", "Jones", "USA", 4, 200)]
columns = ["Prenom", "Nom", "Pays", "ID", "value"]
df = spark.createDataFrame(data, columns)
df.show()
df.printSchema()

## 1. Transformation sans modifier la source
`withColumn` produit un **nouveau** DataFrame. Vérifiez que `df` n’a pas changé.

In [ ]:
df2 = df.withColumnRenamed("value", "montant")
df3 = df2.withColumn("date_reference", F.lit(None).cast("date"))
df3.show()
df.show()

## 2. NULL est-il égal au texte "NULL" ?
On utilise `None` pour une vraie valeur nulle. Remplacez-la par une date arbitraire uniquement pour l’exercice.

In [ ]:
df4 = df3.withColumn("date_reference", F.coalesce(F.col("date_reference"), F.lit("2026-01-01").cast("date")))
df4.show()

## 3. Filtrer puis agréger
Affichez la ligne de l’ID 4. Puis comptez les personnes et additionnez les montants **par pays**.

In [ ]:
df4.filter(F.col("ID") == 4).show()
df4.groupBy("Pays").agg(F.count("*").alias("nb_personnes"), F.sum("montant").alias("total_montant")).show()

## 4. Écrire du Parquet partitionné
Quelles sous-arborescences `Pays=...` attendez-vous ? Attention à la cardinalité : on ne partitionne pas par identifiant unique.

In [ ]:
# IMPORTANT: configure one writable Unity Catalog Volume before this step.
# Use a unique subdirectory to avoid overwriting a classmate's output.
OUTPUT_ROOT = "/Volumes/workspace/default/episen_volume/CHANGE_ME"
assert "CHANGE_ME" not in OUTPUT_ROOT, "Replace CHANGE_ME with your personal identifier."


In [ ]:
path = OUTPUT_ROOT.rstrip("/") + "/tp01_personnes_parquet"
df4.write.mode("overwrite").partitionBy("Pays").parquet(path)
spark.read.parquet(path).show()

## 5. SQL ou API DataFrame ?
Les deux exécutent des traitements Spark. Comparez :

In [ ]:
df4.createOrReplaceTempView("personnes")
spark.sql("SELECT Pays, COUNT(*) AS nb FROM personnes GROUP BY Pays").show()

### Questions de bilan
- Quelle différence entre `filter` et `groupBy` ?
- Pourquoi un vrai `NULL` est-il différent de `"NULL"` ?
- Que gagne-t-on (ou perd-on) en partitionnant des données ?